# Engenharia de Atributos - Pré-processamento de dados - Part 1
> Dados usados: [credit_simple.csv](../anexos/credit_simple.csv)

## Importando os dados

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

dataset = pd.read_csv('../anexos/credit_simple.csv', sep=';')
dataset.shape

(1000, 8)

In [2]:
dataset.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,CLASSE
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv,bom
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv,ruim
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação,bom
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento,bom
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo,ruim


## Separando a Classe das Variáveis Independentes

In [3]:
y = dataset['CLASSE']
X = dataset.iloc[:,:-1]

X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo
...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,29/06/2018,fem div/cas,mobilia/equipamento
996,3857.0,4,40,nenhum,30/06/2018,masculino div/sep,carro usado
997,804.0,4,38,nenhum,03/07/2018,masculino solteiro,radio/tv
998,1845.0,4,23,nenhum,04/07/2019,masculino solteiro,radio/tv


## Verificando valores NULOS

In [4]:
X.isnull().sum()

SALDO_ATUAL         7
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         8
PROPOSITO           0
dtype: int64

## Tratando os valores NULOS - Para Mediana

- Uma boa estratégia é transformar os valores NULOS em medianas;
- Medianas ao contrário da Media, não é impactada por valores discrepantes.

In [5]:
mediana = X['SALDO_ATUAL'].median()
mediana

2323.0

In [6]:
X['SALDO_ATUAL'] = X['SALDO_ATUAL'].fillna(mediana)
X.isnull().sum()

SALDO_ATUAL         0
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         8
PROPOSITO           0
dtype: int64

## Tratando os valores NULOS - Para Moda (valor mais comum)
> No exemplo, a moda será 'masculino solteiro'

In [7]:
agrupado = X.groupby(['ESTADOCIVIL']).size()
agrupado

ESTADOCIVIL
fem div/cas               308
masculino casado/viuvo     92
masculino div/sep          50
masculino solteiro        542
dtype: int64

In [8]:
X['ESTADOCIVIL'] = X['ESTADOCIVIL'].fillna('masculino solteiro')
X.isnull().sum()

SALDO_ATUAL         0
RESIDENCIADESDE     0
IDADE               0
OUTROSPLANOSPGTO    0
DATA                0
ESTADOCIVIL         0
PROPOSITO           0
dtype: int64

## Detectando Outliers

In [9]:
# Uma possível estratégia seria identificar se o valor é >= que 2 desvios padrões
desv = X['SALDO_ATUAL'].std()
desv

685936688.9820064

In [10]:
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL'] # Retorna 2 outliers

127    2.541111e+09
160    2.154441e+10
Name: SALDO_ATUAL, dtype: float64

In [11]:
mediana = X['SALDO_ATUAL'].median() # identifica a mediana
mediana

X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL'] = mediana # substitui os outliers pela mediana
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL']

Series([], Name: SALDO_ATUAL, dtype: float64)

## Aplicando Data Binning

Transforma os dados onde:
- Os valores deveriam ser iguais, mas houve algum erro de digitação;
- Valores diferentes que possuam o mesmo proposito (ex.: moto e motocicleta);
- Há valores quem baixa relevância;

In [12]:
agrupado = X.groupby(['PROPOSITO']).size()
agrupado

PROPOSITO
Eletrodomésticos        12
carro novo             234
carro usado            103
educação                50
mobilia/equipamento    181
negócios                97
obras                   22
outros                  12
qualificação             9
radio/tv               280
dtype: int64

In [13]:
# Agrupando na categoria 'outros'
X.loc[X['PROPOSITO']=='Eletrodomésticos','PROPOSITO'] = 'outros'
X.loc[X['PROPOSITO']=='qualificação','PROPOSITO'] = 'outros'

agrupado = X.groupby(['PROPOSITO']).size()
agrupado

PROPOSITO
carro novo             234
carro usado            103
educação                50
mobilia/equipamento    181
negócios                97
obras                   22
outros                  33
radio/tv               280
dtype: int64

## Extração de categorias
> Exemplo usando o atributo DATA

In [14]:
X['DATA'] # Sem tratamento, é apenas texto. Sem valor de grandeza

0      01/01/2019
1      01/01/2020
2      02/01/2020
3      02/01/2019
4      03/01/2018
          ...    
995    29/06/2018
996    30/06/2018
997    03/07/2018
998    04/07/2019
999    05/07/2018
Name: DATA, Length: 1000, dtype: object

In [15]:
X['DATA'] = pd.to_datetime(X['DATA'], format='%d/%m/%Y')
X['ANO'] = X['DATA'].dt.year
X['MES'] = X['DATA'].dt.month
X['DIA'] = X['DATA'].dt.day
X['DIASEMANA'] = X['DATA'].dt.day_name()

X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,masculino solteiro,radio/tv,2019,1,1,Tuesday
1,5951.0,2,22,nenhum,2020-01-01,fem div/cas,radio/tv,2020,1,1,Wednesday
2,2096.0,3,49,nenhum,2020-01-02,masculino solteiro,educação,2020,1,2,Thursday
3,7882.0,4,45,nenhum,2019-01-02,masculino solteiro,mobilia/equipamento,2019,1,2,Wednesday
4,4870.0,4,53,nenhum,2018-01-03,masculino solteiro,carro novo,2018,1,3,Wednesday


## Label Encoder
> Transformando valores categoricos em númericos

In [16]:
labelencoder = LabelEncoder()
X['ESTADOCIVIL'] = labelencoder.fit_transform(X['ESTADOCIVIL'])
X['PROPOSITO'] = labelencoder.fit_transform(X['PROPOSITO'])
X['DIASEMANA'] = labelencoder.fit_transform(X['DIASEMANA'])

X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,1,5
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,2,4
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,2,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,3,6


## One-hot Encoding (Dummy)
> Criando novos atributos Dummy (0 ou 1) a partir de um atributo categórico.

In [17]:
outros = X['OUTROSPLANOSPGTO'].unique()
outros

array(['nenhum', 'banco', 'stores'], dtype=object)

In [18]:
z = pd.get_dummies(X['OUTROSPLANOSPGTO'], prefix='OUTROS_PLANOS')
z

,OUTROS_PLANOS_banco,OUTROS_PLANOS_nenhum,OUTROS_PLANOS_stores
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False
...,...,...,...
995,False,True,False
996,False,True,False
997,False,True,False
998,False,True,False


## Padronizando valores numéricos (Standard Scaler)
> Para que os atributos não tenham uma influencia desbalanceada para com a classe

Onde fit_transform:
- fit: calcula a média e o desvio padrão de cada coluna selecionada.
- transform: aplica a padronização de escala usando essas estatísticas.

In [ ]:
sc = StandardScaler()
m = sc.fit_transform(X.iloc[:,0:3]) # 'SALDO_ATUAL', 'RESIDENCIADESDE' e 'IDADE'
m

array([[-0.74551643,  1.04698668,  1.6392759 ],
       [ 0.95774038, -0.76597727, -0.74024139],
       [-0.41533679,  0.14050471,  0.68746898],
       ...,
       [-0.87552244,  1.04698668,  0.1058092 ],
       [-0.50473818,  1.04698668, -0.68736323],
       [ 0.46799171,  1.04698668, -0.47585058]], shape=(1000, 3))

## Unificando os dados tratados ao Dataframe original

In [20]:
X = pd.concat([X, z, pd.DataFrame(m, columns=['SALDO_ATUAL_NORMAL', 'RESIDENCIADESDE_NORMAL', 'IDADE_NORMAL'])], axis=1)
X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,DIASEMANA,OUTROS_PLANOS_banco,OUTROS_PLANOS_nenhum,OUTROS_PLANOS_stores,SALDO_ATUAL_NORMAL,RESIDENCIADESDE_NORMAL,IDADE_NORMAL
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,1,5,False,True,False,-0.745516,1.046987,1.639276
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,1,6,False,True,False,0.957740,-0.765977,-0.740241
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,2,4,False,True,False,-0.415337,0.140505,0.687469
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,2,6,False,True,False,1.645526,1.046987,0.475956
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,3,6,False,True,False,0.572709,1.046987,0.898982


## Removendo os dados que não são mais necessários

In [21]:
X.drop(columns=['SALDO_ATUAL','RESIDENCIADESDE', 'IDADE', 'OUTROSPLANOSPGTO', 'OUTROS_PLANOS_banco'], inplace=True) # removendo OUTROS_PLANOS_banco pois One-hot encoding gera uma correlação que pode afetar o modelo se manter todos
X.head()

,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIA,DIASEMANA,OUTROS_PLANOS_nenhum,OUTROS_PLANOS_stores,SALDO_ATUAL_NORMAL,RESIDENCIADESDE_NORMAL,IDADE_NORMAL
0,2019-01-01,3,7,2019,1,1,5,True,False,-0.745516,1.046987,1.639276
1,2020-01-01,0,7,2020,1,1,6,True,False,0.957740,-0.765977,-0.740241
2,2020-01-02,3,2,2020,1,2,4,True,False,-0.415337,0.140505,0.687469
3,2019-01-02,3,3,2019,1,2,6,True,False,1.645526,1.046987,0.475956
4,2018-01-03,3,0,2018,1,3,6,True,False,0.572709,1.046987,0.898982
